In [1]:
from pathlib import Path

import pyvips

# ! if `bipl` loaded before pyvips, breaks pyvips:
# ! cannot load library 'libgobject-2.0-0.dll': error 0x7e.  Additionally, ctypes.util.find_library() did not manage to locate a library called 'libgobject-2.0-0.dll'
import bipl
from bipl.ops import get_fusion
from glow import starmap_n

from tqdm.auto import tqdm

slides = Path('E:/datasets/ckb/slides')
assert slides.is_dir()

alt = slides / 'alt'
alt.mkdir(exist_ok=True)
# TODO: icc

C:\Users\pmaevskikh\Sources\bipl\src\bipl\io\_slide.py:36: UserWarning: No GDAL is available. Please acquire it manually from https://github.com/cgohlke/geospatial-wheels/releases 
  warn(msg, stacklevel=1)


In [2]:
ZOOM = 2


def iter_tasks(slides: Path):
    for p in tqdm([*slides.iterdir()], desc='scheduling'):
        if p.suffix not in ('.mrxs', '.svs', '.tiff', '.tif', '.ndpi'):
            continue
        s = bipl.Slide.open(p)
        if (mpp := s.spacing) is None:
            tqdm.write(f'No MPP: {s.path}')
            continue

        mpps = [mpp * p for p in s.pools]
        if any(0.5 <= m <= 1.0 for m in mpps):
            continue
        res = 1000 / mpp
        yield p, s, res


def process(p: Path, s: bipl.Slide, res: float):
    path_2 = alt / p.with_suffix('.tif').name
    if False:
        im = pyvips.Image.new_from_file(p, access='sequential')
        if True:
            im2 = im.resize(1 / ZOOM)
        elif False:
            im2 = im.shrink(ZOOM, ZOOM)
        elif False:
            im2 = im.reduce(ZOOM, ZOOM)
    elif False:  # works fast enough
        sp = s.pool(ZOOM)
        tqdm.write(f'- {sp.shape}')
        im2 = pyvips.Image.new_from_array(
            get_fusion(
                bipl.Mosaic(1024, 0).iterate(sp, max_workers=4),
                sp.shape[:2],
            )
        )
    else:
        tqdm.write(f'- {sp.shape}')
        im2 = pyvips.Image.new_from_array(s.pool(ZOOM).numpy())
    tqdm.write(f'- loaded: {im2}')
    im2.tiffsave(
        path_2.as_posix(),
        compression='jpeg',
        Q=85,
        tile=True,
        tile_width=256,
        tile_height=256,
        pyramid=True,
        xres=res / ZOOM,
        yres=res / ZOOM,
        premultiply=True,
    )
    tqdm.write(f'- saved: {path_2}')


tasks = list(iter_tasks(slides))
for _ in tqdm(
    starmap_n(process, tasks, max_workers=4, order=False),
    total=len(tasks),
    desc='processing',
):
    pass

scheduling:   0%|          | 0/811 [00:00<?, ?it/s]

scheduling: 100%|██████████| 811/811 [00:14<00:00, 56.67it/s]


- (41062, 96612, 3)
- (38210, 62748, 3)
- (40286, 64740, 3)
- (40424, 81672, 3)


processing:   0%|          | 0/728 [01:58<?, ?it/s]

- loaded: <pyvips.Image 62748x38210 uchar, 3 bands, srgb>


...

processing: 100%|█████████▉| 726/728 [8:16:07<01:13, 36.63s/it]

- saved: E:\datasets\ckb\slides\alt\9142-53 [2013] PROST-001.tif


processing: 100%|█████████▉| 727/728 [8:18:39<00:59, 59.05s/it]

- saved: E:\datasets\ckb\slides\alt\9908-19 [2013] PROST.tif


processing: 100%|██████████| 728/728 [8:18:40<00:00, 41.10s/it]


In [3]:
import cv2
import matplotlib.pyplot as plt

tsize = 512
g = 5
z = 64 / s.spacing
ims = cv2.cvtColor(s.pool(z).numpy(), cv2.COLOR_RGB2GRAY)
_, ims = cv2.threshold(ims, 200, 255, cv2.THRESH_BINARY_INV)
ims = cv2.GaussianBlur(ims, (g, g), 0)
ims[:g, :] = ims[-g:, :] = 0
ims[:, :g] = ims[:, -g:] = 0
y, x = (*(round(a * z) for a in divmod(ims.argmax(), ims.shape[1])),)

s2 = bipl.Slide.open(path_2)
print(s)
print(s2)

_, (a0, a1) = plt.subplots(1, 2, figsize=(10, 10))
a0.imshow(s.at((y, x), tsize, scale=0.5))
a1.imshow(s2.at((y // 2, x // 2), tsize, scale=1))
a0.set(xticks=[], yticks=[])
a1.set(xticks=[], yticks=[])
plt.tight_layout()
plt.show()

NameError: name 's' is not defined